In [ ]:
# 1. Import necessary packages
import os
import numpy as np
import PIL
import pyvista as pv

In [ ]:
# 2. Import files
import os
import numpy as np
from PIL import Image

folder_path = r'E:\working_stack\WT10 0.5mm_72h'
image_set = []

# 获取文件夹中所有 png 图片，按文件名排序
file_list = sorted(f for f in os.listdir(folder_path) if f.lower().endswith('.png'))

for file_name in file_list:
    img = np.array(Image.open(os.path.join(folder_path, file_name)))
    image_set.append(img)

print(f'共导入 {len(image_set)} 张图片')
print(f'第一张图片尺寸: {image_set[0].shape}')

共导入 59 张图片
第一张图片尺寸: (3571, 5113, 3)


In [6]:
print(f'前10张图片尺寸:')
for i in range(0,10):
   print(f'{image_set[i].shape}')


前10张图片尺寸:
(3571, 5113, 3)
(4045, 5668, 3)
(3928, 5547, 3)
(3384, 4878, 3)
(3184, 4342, 3)
(3665, 5202, 3)
(3471, 4958, 3)
(4165, 5812, 3)
(4182, 5948, 3)
(4254, 6363, 3)


In [ ]:
# 3. Warping the images to same size
from PIL import Image

# 以第一张图片的尺寸为标准 (H, W)
target_h, target_w = image_set[0].shape[0], image_set[0].shape[1]
target_size = (target_w, target_h)  # PIL 的尺寸格式是 (W, H)

for i in range(len(image_set)):
    img_pil = Image.fromarray(image_set[i])
    img_resized = img_pil.resize(target_size, Image.LANCZOS)
    image_set[i] = np.array(img_resized)

print(f'统一后的尺寸: {image_set[0].shape}')
print(f'共 {len(image_set)} 张图片已缩放')

统一后的尺寸: (3571, 5113, 3)
共 59 张图片已缩放


In [8]:
print(f'前10张图片尺寸:')
for i in range(0,10):
   print(f'{image_set[i].shape}')


前10张图片尺寸:
(3571, 5113, 3)
(3571, 5113, 3)
(3571, 5113, 3)
(3571, 5113, 3)
(3571, 5113, 3)
(3571, 5113, 3)
(3571, 5113, 3)
(3571, 5113, 3)
(3571, 5113, 3)
(3571, 5113, 3)


In [ ]:
# 4. Extracting RGB status of every image， with the scale of single pixel
# 将前10张图片的像素 RGB 值导出为数组，按像素坐标顺序排列
# 每张图片单独一个数组（避免单个数组过大），形状为 (H*W, 3)
# 像素顺序：先行后列，即 (0,0), (0,1), ..., (0,W-1), (1,0), (1,1), ...

rgb_arrays = []

for i in range(10):
    # reshape 成 (像素总数, 3)，每行为一个像素的 [R, G, B]
    rgb = image_set[i].reshape(-1, 3)
    rgb_arrays.append(rgb)
    print(f'第 {i+1}个图片: 形状 {rgb.shape}, 数据类型 {rgb.dtype}')

print(f'\n共创建 {len(rgb_arrays)} 个数组')
print(f'像素 (0,0) 的 RGB: {rgb_arrays[0][0]}')
print(f'像素 (0,1) 的 RGB: {rgb_arrays[0][1]}')
print(f'像素 (1,0) 的 RGB: {rgb_arrays[0][11451400]}') 

第 1个图片: 形状 (18258523, 3), 数据类型 uint8
第 2个图片: 形状 (18258523, 3), 数据类型 uint8
第 3个图片: 形状 (18258523, 3), 数据类型 uint8
第 4个图片: 形状 (18258523, 3), 数据类型 uint8
第 5个图片: 形状 (18258523, 3), 数据类型 uint8
第 6个图片: 形状 (18258523, 3), 数据类型 uint8
第 7个图片: 形状 (18258523, 3), 数据类型 uint8
第 8个图片: 形状 (18258523, 3), 数据类型 uint8
第 9个图片: 形状 (18258523, 3), 数据类型 uint8
第 10个图片: 形状 (18258523, 3), 数据类型 uint8

共创建 10 个数组
像素 (0,0) 的 RGB: [0 0 0]
像素 (0,1) 的 RGB: [0 0 0]
像素 (1,0) 的 RGB: [34  0  0]


In [ ]:
# 5. 切片堆叠 3D 重构 —— 动态插值
# 核心思路：用生成器逐张产出切片（原始 + 插值），内存同一时刻最多保留 2 张原图，
#          插值切片即用即弃，避免一次性物化全部 37 张切片
import os
import numpy as np
from PIL import Image
import pyvista as pv

# ---------------- 参数 ----------------
FOLDER = r'E:\working_stack\WT10 0.5mm_72h'
N_SLICES = 10                    # 使用前 10 张图片（从上到下）
Z_SPACING = 4.0                  # 相邻两张原图的 z 间距（数据坐标）
T_WEIGHTS = (0.25, 0.50, 0.75)   # 插值权重 t
RENDER_SCALE = 4                 # 渲染降采样倍数（减小显存占用；1 = 原始分辨率）
Z_DISPLAY_SCALE = 15             # 仅影响显示：z 拉伸倍数（否则 36 单位厚的堆叠在 5000 单位宽的图像旁薄如纸）
SLICE_OPACITY = 0.35             # 切片不透明度（调小 = 更透明的"体积"效果）
OUTPUT_PNG = r'E:\SRT\stack3d.png'

# ---------------- 惰性读取：从磁盘逐张加载，同一时刻只保留一张 ----------------
def lazy_load_slices(folder, n):
    """按文件名排序逐张读取 png，统一到第一张的尺寸；不一次性载入全部图片"""
    files = sorted(f for f in os.listdir(folder) if f.lower().endswith('.png'))[:n]
    with Image.open(os.path.join(folder, files[0])) as ref:
        target_size = ref.size                      # (W, H)
    for name in files:
        with Image.open(os.path.join(folder, name)) as im:
            im = im.convert('RGB')
            if im.size != target_size:
                im = im.resize(target_size, Image.LANCZOS)
            yield np.array(im)

# ---------------- 切片堆叠生成器 ----------------
def slice_stack_generator(image_iter, z_spacing=Z_SPACING, t_weights=T_WEIGHTS):
    """
    产出 (z, image) 序列，z 轴递减：
      原始切片：z_i = -i * z_spacing
      插值切片：I = t*I(z_i) + (1-t)*I(z_{i+1})，位于 z = t*z_i + (1-t)*z_{i+1}
    """
    prev = next(image_iter, None)
    i = 0
    while prev is not None:
        z_i = -i * z_spacing
        curr = next(image_iter, None)
        yield z_i, prev                             # 原始切片
        if curr is not None:                        # 与下一张之间的 3 张插值切片
            a = prev.astype(np.float32)
            b = curr.astype(np.float32)
            z_next = -(i + 1) * z_spacing
            for t in t_weights:
                interp = np.clip((1-t) * a + t * b, 0, 255).astype(np.uint8)
                yield (1-t) * z_i + t * z_next, interp
            del a, b
        prev = curr
        i += 1

# ---------------- 验证：只遍历统计，不存储任何切片 ----------------
zs = []
shape = None
for z, img in slice_stack_generator(lazy_load_slices(FOLDER, N_SLICES)):
    zs.append(z)
    shape = img.shape
print(f'共生成 {len(zs)} 张切片（{N_SLICES} 张原图 + {len(zs) - N_SLICES} 张插值）')
print(f'单张切片尺寸: {shape}')
print(f'z 范围: [{min(zs):.0f}, {max(zs):.0f}]，是否均匀间隔 1 个单位: {np.allclose(np.diff(sorted(zs)), 1.0)}')
print(f'前 8 个 z（生成顺序）: {[round(v, 2) for v in zs[:8]]}')

共生成 37 张切片（10 张原图 + 27 张插值）
单张切片尺寸: (3571, 5113, 3)
z 范围: [-36, 0]，是否均匀间隔 1 个单位: True
前 8 个 z（生成顺序）: [0.0, -1.0, -2.0, -3.0, -4.0, -5.0, -6.0, -7.0]


In [7]:
# 6. 渲染切片堆叠（离屏渲染 → 导出静态 PNG）
# 每张切片 = 一个带纹理的平面；纹理上传 GPU 后原图数组即被丢弃（动态渲染，不存储全部图片）
import numpy as np
from PIL import Image
import pyvista as pv

plotter = pv.Plotter(off_screen=True, window_size=(1920, 1440))
plotter.set_background('black')
# 注意：不要使用 enable_depth_peeling()！
# 在当前离屏渲染环境下深度剥离不可用，会导致输出全黑（已实测验证）

n_added = 0
for z, img in slice_stack_generator(lazy_load_slices(FOLDER, N_SLICES)):
    # 渲染降采样：大幅减小纹理显存占用（RENDER_SCALE=4 → 约 893×1278）
    h, w = img.shape[0] // RENDER_SCALE, img.shape[1] // RENDER_SCALE
    # img_small = np.array(Image.fromarray(img).resize((w, h), Image.BILINEAR))

    # 平面位于显示坐标 z * Z_DISPLAY_SCALE；平面边长取像素数，保持像素纵横比 1:1
    plane = pv.Plane(center=(0, 0, z * Z_DISPLAY_SCALE),
                     direction=(0, 0, 1), i_size=w, j_size=h)
    plane.texture_map_to_plane(inplace=True)      # 生成 UV 纹理坐标

    # VTK 纹理 v=0 位于底部，而图像第 0 行在顶部 → 垂直翻转一次以保持正立
    tex = pv.numpy_to_texture(np.ascontiguousarray(img[::-1]))
    plotter.add_mesh(plane, texture=tex, opacity=SLICE_OPACITY)

    n_added += 1
    del img                           # 即用即弃，控制内存

print(f'已添加 {n_added} 张切片平面')
plotter.camera_position = [
    (-5500, -3800, 2000),    # 相机位置：负 x、负 y 象限，略高于堆叠
    (0, 0, -270),            # 焦点：堆叠几何中心
    (0, 0, 1)                # 上方向
]
plotter.screenshot(OUTPUT_PNG)
print(f'渲染完成，已保存至 {OUTPUT_PNG}')

已添加 37 张切片平面
渲染完成，已保存至 E:\SRT\stack3d.png
